In [5]:
from dotenv import load_dotenv
import os
load_dotenv()
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain.chat_models import init_chat_model
api_key = os.getenv("DEEPSEEK_API_KEY")

model = init_chat_model(
    model="deepseek-chat",
    model_provider="openai",
    base_url="https://api.deepseek.com",
    api_key=api_key
)

In [6]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.runnables import RunnableConfig

# 初始化checkpointer
checkpointer = InMemorySaver()
# 初始化中间件
middleware = SummarizationMiddleware(
    model=model,
    trigger=("messages", 3), # 触发时机，当消息数超过3时，进行总结
    keep=("messages", 1) # 保留的会话数，超过2条
)
# 创建agent
agent = create_agent(model=model,checkpointer=checkpointer,middleware=[middleware])


In [8]:
from langchain_core.runnables import RunnableConfig

config: RunnableConfig = {"configurable": {"thread_id": "thread_3"}}
# 制造长会话历史
agent.invoke({"messages": [HumanMessage(content="你好，我是虎哥。")]}, config)
agent.invoke({"messages": [HumanMessage(content="我最喜欢的运动是乒乓")]}, config)
agent.invoke({"messages": [HumanMessage(content="我最喜欢的动物是猫猫")]}, config)
# 测试效果
final_response = agent.invoke({"messages": HumanMessage(content="你还记得我吗？")}, config)
print(final_response)

{'messages': [HumanMessage(content='Here is a summary of the conversation to date:\n\n## SESSION INTENT\n用户（虎哥）正在进行自我介绍/闲聊，尚未提出具体工作任务或实质性请求。助手正通过提问延续对话、加深了解，目标是推动对话进入实质性任务。\n\n## SUMMARY\n- 用户自称“虎哥”，开场为简短问候/自我介绍（“你好，我是虎哥。”）。\n- 用户已分享的个人信息：最喜欢的运动是乒乓球（“我最喜欢的运动是乒乓”）；最喜欢的动物是猫猫（多次强调，最近一次重申“我最喜欢的动物是猫猫”）。\n- 用户曾问“你还记得我吗？”，助手已确认记得并复述了乒乓球、猫猫两项信息。\n- 助手已对乒乓球提问参与程度（娱乐/系统训练/比赛）与打法偏好（进攻/防守）；对猫猫提问自己养/单纯喜欢、猫的性格偏好（活泼好动/安静黏人）。用户均未回答，仅重复了喜欢猫猫。\n- 尚未确定任何具体目标、策略或决策；无被拒绝的选项。对话仍处于建立了解阶段。\n\n## ARTIFACTS\nNone\n\n## NEXT STEPS\n继续等待用户（虎哥）回答：乒乓球参与程度与打法偏好、养猫情况与猫的性格偏好。可考虑变换提问方式（如减少问题数量、先聊猫猫单一话题）以降低回答门槛，据此继续了解其需求，推动对话进入实质性任务。', additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='38093e98-7998-4f79-aeac-1d74d5441615'), HumanMessage(content='你还记得我吗？', additional_kwargs={}, response_metadata={}, id='09dd5892-22e9-493d-8da8-8f2e22331eb9'), AIMessage(content='当然记得，虎哥！你喜欢打乒乓球，最喜欢的动物是猫猫。😊\n\n对了，上次我还想多了解你一点——你是自己养猫，还是单纯喜欢猫呀？', additional_kwargs={'refusal': None}, response_metadata={'token_usage':